**Cell 1: Create Catalog and Schemas**

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS adb_databricks_project_dev;

CREATE SCHEMA IF NOT EXISTS adb_databricks_project_dev.metadata;

CREATE SCHEMA IF NOT EXISTS adb_databricks_project_dev.bronze;

CREATE SCHEMA IF NOT EXISTS adb_databricks_project_dev.silver;

CREATE SCHEMA IF NOT EXISTS adb_databricks_project_dev.gold;

**Cell 2: Create Managed Volume**

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS
adb_databricks_project_dev.metadata.landing_volume;

**Cell 3: Create Folder Structure**

In [0]:
base_path = (
    "/Volumes/"
    "adb_databricks_project_dev/"
    "metadata/"
    "landing_volume"
)

dbutils.fs.mkdirs(base_path + "/incoming/Order")
dbutils.fs.mkdirs(base_path + "/archive/Order")
dbutils.fs.mkdirs(base_path + "/rejected/Order")

print("Folder structure created successfully")

**Cell 4: Create Source Configuration Table**

In [0]:
%sql

CREATE TABLE IF NOT EXISTS adb_databricks_project_dev.metadata.source_config
(
    ------------------------------------------------------------------
    -- Source Information
    ------------------------------------------------------------------
    source_id STRING,
    entity_name STRING,
    source_name STRING,

    ------------------------------------------------------------------
    -- Source Type
    ------------------------------------------------------------------
    source_type STRING,              -- FILE / DATABASE / REST_API / EXCEL

    ------------------------------------------------------------------
    -- Source Location
    ------------------------------------------------------------------
    source_path STRING,

    ------------------------------------------------------------------
    -- Future Database Source
    ------------------------------------------------------------------
    source_table STRING,
    source_query STRING,
    connection_name STRING,

    ------------------------------------------------------------------
    -- Future REST API Source
    ------------------------------------------------------------------
    api_url STRING,
    api_root_key STRING,
    api_multiline STRING,

    ------------------------------------------------------------------
    -- Target Information
    ------------------------------------------------------------------
    target_catalog STRING,

    bronze_schema STRING,
    silver_schema STRING,
    gold_schema STRING,

    bronze_table STRING,
    silver_table STRING,
    gold_table STRING,

    ------------------------------------------------------------------
    -- Entity Group
    ------------------------------------------------------------------
    bronze_group STRING,

    ------------------------------------------------------------------
    -- Load Configuration
    ------------------------------------------------------------------
    load_type STRING,
    watermark_column STRING,

    ------------------------------------------------------------------
    -- File Read Options
    --
    -- NOTE:
    -- File format is NOT stored here.
    -- The Bronze notebook automatically detects the file format
    -- from the file extension at runtime.
    ------------------------------------------------------------------

    delimiter STRING,
    header_flag STRING,
    sheet_name STRING,

    ------------------------------------------------------------------
    -- Archive Locations
    ------------------------------------------------------------------
    archive_path STRING,
    reject_path STRING,

    ------------------------------------------------------------------
    -- Status
    ------------------------------------------------------------------
    active_flag STRING

)
USING DELTA;

**Cell 5: Create File Process Log Table**

In [0]:
%sql

CREATE TABLE IF NOT EXISTS
adb_databricks_project_dev.metadata.file_process_log
(
    source_id STRING,
    entity_name STRING,
    source_type STRING,
    file_format STRING,
    file_name STRING,
    file_path STRING,
    target_table STRING,
    status STRING,
    load_time TIMESTAMP,
    error_message STRING
)
USING DELTA;

**Cell 6: Create Execution Log Table**

In [0]:
%sql

CREATE TABLE IF NOT EXISTS
adb_databricks_project_dev.metadata.execution_log
(
    execution_id STRING,
    entity_name STRING,
    source_type STRING,
    file_format STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    files_processed BIGINT,
    rows_loaded BIGINT,
    error_message STRING
)
USING DELTA;

**Cell 7: Create Silver Transformation Config Table**

In [0]:
%sql
CREATE TABLE adb_databricks_project_dev.metadata.transformation_config
(
    ------------------------------------------------------------------
    -- Transformation Information
    ------------------------------------------------------------------
    transformation_id STRING,

    entity_name STRING,

    ------------------------------------------------------------------
    -- Source & Target
    ------------------------------------------------------------------
    source_table STRING,

    target_table STRING,

    ------------------------------------------------------------------
    -- Column Mapping
    ------------------------------------------------------------------
    source_column STRING,

    target_column STRING,

    ------------------------------------------------------------------
    -- Data Type
    ------------------------------------------------------------------
    target_data_type STRING,

    ------------------------------------------------------------------
    -- Transformation Rule
    ------------------------------------------------------------------
    transformation_rule STRING,

    ------------------------------------------------------------------
    -- Merge Information
    ------------------------------------------------------------------
    primary_key STRING,

    rule_order INT,

    deduplicate_flag STRING,

    ------------------------------------------------------------------
    -- Status
    ------------------------------------------------------------------
    active_flag STRING

)
USING DELTA;

**Cell 9: Insert Entry in Source Metadata**

In [0]:
%sql

INSERT INTO adb_databricks_project_dev.metadata.source_config
(
    ------------------------------------------------------------------
    -- Source Information
    ------------------------------------------------------------------
    source_id,
    entity_name,
    source_name,

    ------------------------------------------------------------------
    -- Source Type
    ------------------------------------------------------------------
    source_type,

    ------------------------------------------------------------------
    -- Source Location
    ------------------------------------------------------------------
    source_path,

    ------------------------------------------------------------------
    -- Database Source
    ------------------------------------------------------------------
    source_table,
    source_query,
    connection_name,

    ------------------------------------------------------------------
    -- REST API Source (Future)
    ------------------------------------------------------------------
    api_url,
    api_root_key,
    api_multiline,

    ------------------------------------------------------------------
    -- Target Information
    ------------------------------------------------------------------
    target_catalog,

    bronze_schema,
    silver_schema,
    gold_schema,

    bronze_table,
    silver_table,
    gold_table,

    ------------------------------------------------------------------
    -- Bronze Group
    ------------------------------------------------------------------
    bronze_group,

    ------------------------------------------------------------------
    -- Load Configuration
    ------------------------------------------------------------------
    load_type,
    watermark_column,

    ------------------------------------------------------------------
    -- File Options
    ------------------------------------------------------------------
    delimiter,
    header_flag,
    sheet_name,

    ------------------------------------------------------------------
    -- Archive Paths
    ------------------------------------------------------------------
    archive_path,
    reject_path,

    ------------------------------------------------------------------
    -- Status
    ------------------------------------------------------------------
    active_flag
)

VALUES

(
    ------------------------------------------------------------------
    -- Order Entity
    ------------------------------------------------------------------
    '1',

    'Order',
    'Order File Source',

    'FILE',

    '/Volumes/adb_databricks_project_dev/metadata/landing_volume/incoming/Order',

    NULL,
    NULL,
    NULL,

    NULL,
    NULL,
    NULL,

    'adb_databricks_project_dev',

    'bronze',
    'silver',
    'gold',

    'order',
    'order',
    'order_summary',

    'SALES',

    'APPEND',
    NULL,

    ',',
    'Y',
    NULL,

    '/Volumes/adb_databricks_project_dev/metadata/landing_volume/archive/Order',

    '/Volumes/adb_databricks_project_dev/metadata/landing_volume/rejected/Order',

    'Y'
),

(
    ------------------------------------------------------------------
    -- Customer Entity
    ------------------------------------------------------------------
    '2',

    'Customer',
    'Customer File Source',

    'FILE',

    '/Volumes/adb_databricks_project_dev/metadata/landing_volume/incoming/Customer',

    NULL,
    NULL,
    NULL,

    NULL,
    NULL,
    NULL,

    'adb_databricks_project_dev',

    'bronze',
    'silver',
    'gold',

    'customer',
    'customer',
    'customer_summary',

    'MASTER',

    'APPEND',
    NULL,

    ',',
    'Y',
    NULL,

    '/Volumes/adb_databricks_project_dev/metadata/landing_volume/archive/Customer',

    '/Volumes/adb_databricks_project_dev/metadata/landing_volume/rejected/Customer',

    'Y'
);

**Cell 10: Insert Order Transformation Metadata**

**Cell 11: Verify Source Metadata**

In [0]:
%sql

SELECT
    source_id,
    entity_name,
    source_name,
    source_type,
    file_format,
    bronze_table,
    active_flag
FROM adb_databricks_project_dev.metadata.source_config
WHERE entity_name = 'Order'
ORDER BY source_id;

**Cell 12: Verify Transformation Metadata**

In [0]:
%sql

SELECT
    transformation_id,
    entity_name,
    source_type,
    file_format,
    transformation_type,
    primary_key,
    deduplicate_flag,
    active_flag
FROM adb_databricks_project_dev.metadata.transformation_config
WHERE entity_name = 'Order'
ORDER BY transformation_id;

**Cell 13: Verify Folder Structure**

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/adb_databricks_project_dev/metadata/landing_volume"
    )
)

display(
    dbutils.fs.ls(
        "/Volumes/adb_databricks_project_dev/metadata/landing_volume/incoming/Order"
    )
)

In [0]:
%sql
UPDATE adb_databricks_project_dev.metadata.source_config
SET active_flag='N'
WHERE source_type='REST_API';